# 02 · Model quality on four dimensions

Computes **fitness, precision, generalization and simplicity** for Alpha,
Heuristic and Inductive Miner, under **both case notions**, for every available
faculty and role.

The main analysis uses explicit miner variants for reproducibility:

* Alpha Miner: PM4Py default Alpha implementation
* Heuristic Miner: `Variants.CLASSIC`
* Inductive Miner: `Variants.IM` (plain IM)

The separate parameter-sensitivity notebook (`04_parameter_sensitivity.ipynb`)
uses `Variants.IMf` only for the Inductive Miner noise-threshold sweep.

### Memory warning

ETC precision builds a prefix automaton whose size scales with **trace length**.
Two guards are applied below:

* `SAMPLE_TRACES_COURSE` / `SAMPLE_TRACES_USER` cap evaluated traces
* `MAX_TRACE_LEN` excludes traces longer than the declared threshold

The same sampled log is used for all three miners within each
faculty-role-case-notion comparison. Sampling uses the fixed `SEED`, and the
chosen values are written to `02_run_metadata.json`.


## 1. Setup

Run this section first. It installs dependencies and downloads the deposit from
figshare into the Colab VM.

**Runtime:** Runtime &rarr; Change runtime type &rarr; **High-RAM** if available.
The largest faculty file (FIF, 1.3 GB on disk) needs roughly 6 GB once loaded.

In [ ]:
#@title Install dependencies { display-mode: "form" }
!pip install -q pm4py==2.7.23.3 statsmodels 2>/dev/null
import os
os.environ["TQDM_DISABLE"] = "1"

import warnings, sys, json, time, gc, random
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np

print("Python ", sys.version.split()[0])
print("pandas ", pd.__version__)
import pm4py; print("pm4py  ", pm4py.__version__)

# --- RAM report -------------------------------------------------------------
try:
    import psutil
    gb = psutil.virtual_memory().total / 1e9
    print(f"RAM    {gb:.1f} GB")
    if gb < 20:
        print("\\n  NOTE: standard runtime. FIF and FTE may run out of memory.")
        print("  Runtime -> Change runtime type -> High-RAM is recommended.")
except Exception:
    pass

In [ ]:
#@title Download the deposit from figshare { display-mode: "form" }
# Queries the figshare API, so lecturer files are picked up automatically
# once they are added to the deposit.

import requests, os, pathlib

ARTICLE = "28341992"          #@param {type:"string"}
DATA_DIR = "/content/data"    #@param {type:"string"}
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

meta = requests.get(f"https://api.figshare.com/v2/articles/{ARTICLE}", timeout=60).json()
print(f"{meta['title']}  (v{meta.get('version','?')})")
print(f"{len(meta['files'])} files, {meta['size']/1e9:.2f} GB total\n")

FILES = {}
for f in meta["files"]:
    FILES[f["name"]] = f["download_url"]
    print(f"  {f['name']:<32} {f['size']/1e6:>8.1f} MB")

# --- completeness check -----------------------------------------------------
FACULTIES = ["FEB", "FIF", "FIK", "FIT", "FKB", "FRI", "FTE"]
missing = [f"{fac}_{role}.csv" for fac in FACULTIES
           for role in ("Student", "Lecturer") if f"{fac}_{role}.csv" not in FILES]
if missing:
    print("\n  MISSING FROM DEPOSIT:")
    for m in missing:
        print(f"    {m}")
    print("\n  Analyses for these partitions will be skipped.")


def fetch(name):
    """Download one file if not already present. Returns local path or None."""
    if name not in FILES:
        return None
    dest = os.path.join(DATA_DIR, name)
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        return dest
    print(f"downloading {name} ...", flush=True)
    with requests.get(FILES[name], stream=True, timeout=1800) as r:
        r.raise_for_status()
        with open(dest, "wb") as fh:
            for chunk in r.iter_content(1 << 22):
                fh.write(chunk)
    print(f"  -> {os.path.getsize(dest)/1e6:.0f} MB")
    return dest

In [ ]:
#@title Loader { display-mode: "form" }
# Memory-efficient reader plus the two corrections identified during revision.

USECOLS = ["id", "eventname", "component", "action", "target", "crud",
           "edulevel", "userid", "courseid", "timecreated", "event"]
DTYPES = {"id": "int64", "eventname": "category", "component": "category",
          "action": "category", "target": "category", "crud": "category",
          "edulevel": "int8", "userid": "int32", "courseid": "int32",
          "event": "category"}

CUTOFF = "2023-06-26"   # verified coverage boundary (NOT July, see manuscript)


def load(faculty, role, apply_dedup=True, cols=None):
    """Load one faculty-role partition.

    apply_dedup fixes the defect found during revision: the original notebooks
    called df.drop_duplicates() WITHOUT assignment, so duplicates were counted
    and reported but never removed from the working data.
    """
    name = f"{faculty}_{role}.csv"
    path = fetch(name)
    if path is None:
        print(f"  [skip] {name} not in deposit")
        return None
    use = cols or USECOLS
    df = pd.read_csv(path, index_col=0, usecols=lambda c: c in use or c == "Unnamed: 0",
                     dtype={k: v for k, v in DTYPES.items() if k in use},
                     parse_dates=["timecreated"] if "timecreated" in use else None)
    n_raw = len(df)
    n_dup = int(df.duplicated().sum())
    if apply_dedup and n_dup:
        df = df.drop_duplicates()          # assignment: this is the fix
    df.attrs["n_raw"] = n_raw
    df.attrs["n_dup"] = n_dup
    df.attrs["faculty"] = faculty
    df.attrs["role"] = role
    return df


def add_case(df, notion):
    """notion is 'user' or 'course'."""
    if notion == "user":
        df["case"] = df["userid"].astype(str)
    else:
        df["case"] = df["userid"].astype(str) + "_" + df["courseid"].astype(str)
    return df


def to_log(df, notion, sample=None, seed=42, max_len=None):
    """Build a pm4py EventLog. sample caps the number of traces."""
    d = add_case(df, notion)
    if max_len:
        keep = d.groupby("case").size()
        d = d[d["case"].isin(keep[keep <= max_len].index)]
    if sample:
        random.seed(seed)
        cases = sorted(d["case"].unique())
        d = d[d["case"].isin(set(random.sample(cases, min(sample, len(cases)))))]
    ldf = (d[["case", "event", "timecreated"]]
           .rename(columns={"case": "case:concept:name", "event": "concept:name",
                            "timecreated": "time:timestamp"})
           .sort_values(["case:concept:name", "time:timestamp"])
           .reset_index(drop=True))
    # pm4py rejects categorical columns: cast the two key columns to str
    ldf["case:concept:name"] = ldf["case:concept:name"].astype(str)
    ldf["concept:name"] = ldf["concept:name"].astype(str)
    return pm4py.convert_to_event_log(ldf), ldf


def save(obj, name):
    """Persist a result table and offer it for download."""
    if isinstance(obj, pd.DataFrame):
        obj.to_csv(f"/content/{name}.csv", index=False)
    else:
        json.dump(obj, open(f"/content/{name}.json", "w"), indent=1)
    print(f"saved /content/{name}")

## 2. Configuration

In [ ]:
#@title Run parameters { display-mode: "form" }
SAMPLE_TRACES_COURSE = 300   #@param {type:"integer"}
SAMPLE_TRACES_USER   = 40    #@param {type:"integer"}
MAX_TRACE_LEN        = 2000  #@param {type:"integer"}
SEED                 = 42    #@param {type:"integer"}
RUN_FACULTIES = ["FIT", "FKB", "FIK", "FRI", "FEB", "FTE", "FIF"]  # cheapest first
RUN_ROLES     = ["Student", "Lecturer"]
RUN_NOTIONS   = ["course", "user"]

from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.discovery.heuristics import algorithm as heuristics_miner
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.algo.evaluation.replay_fitness import algorithm as fit_eval
from pm4py.algo.evaluation.precision import algorithm as prec_eval
from pm4py.algo.evaluation.generalization import algorithm as gen_eval
from pm4py.algo.evaluation.simplicity import algorithm as sim_eval

P = {"show_progress_bar": False}

MINER_VARIANTS = {
    "alpha": "PM4Py Alpha default",
    "heuristic": "Variants.CLASSIC",
    "inductive": "Variants.IM",
}


def discover(log, miner):
    if miner == "alpha":
        return alpha_miner.apply(log)
    if miner == "heuristic":
        return heuristics_miner.apply(
            log, variant=heuristics_miner.Variants.CLASSIC)
    t = inductive_miner.apply(
        log, variant=inductive_miner.Variants.IM)
    return pm4py.convert_to_petri_net(t) if not isinstance(t, tuple) else t


metadata = {
    "python": sys.version,
    "pandas": pd.__version__,
    "pm4py": pm4py.__version__,
    "sample_traces_course": SAMPLE_TRACES_COURSE,
    "sample_traces_user": SAMPLE_TRACES_USER,
    "max_trace_len": MAX_TRACE_LEN,
    "seed": SEED,
    "miner_variants": MINER_VARIANTS,
    "analysis_window_note": "Deposited role partitions are already restricted to dates before 2023-06-26."
}
save(metadata, "02_run_metadata")
print(json.dumps(metadata, indent=2))


## 3. Run

Writes each result to `/content/02_quality.csv` as it goes, so a disconnect does
not lose completed work. Re-running skips combinations already present.

In [ ]:
import os
OUT = "/content/02_quality.csv"
done = set()
if os.path.exists(OUT):
    prev = pd.read_csv(OUT)
    done = {(r.faculty, r.role, r.notion, r.miner) for r in prev.itertuples()}
    print(f"resuming: {len(done)} combinations already computed")

results = []


def flush():
    d = pd.DataFrame(results)
    if os.path.exists(OUT):
        d = pd.concat([pd.read_csv(OUT), d], ignore_index=True)
    d.drop_duplicates(subset=["faculty", "role", "notion", "miner"],
                      keep="last").to_csv(OUT, index=False)


for fac in RUN_FACULTIES:
    for role in RUN_ROLES:
        df = None
        for notion in RUN_NOTIONS:
            todo = [m for m in ("alpha", "heuristic", "inductive")
                    if (fac, role, notion, m) not in done]
            if not todo:
                continue
            if df is None:
                df = load(fac, role)
                if df is None:
                    break
            n = SAMPLE_TRACES_COURSE if notion == "course" else SAMPLE_TRACES_USER
            log, ldf = to_log(df, notion, sample=n, seed=SEED, max_len=MAX_TRACE_LEN)
            tl = ldf.groupby("case:concept:name").size()
            print(f"\n[{fac} {role} {notion}] {len(log):,} traces, "
                  f"{len(ldf):,} events, mean len {tl.mean():.0f}", flush=True)

            for miner in todo:
                r = dict(faculty=fac, role=role, notion=notion, miner=miner,
                         traces=len(log), events=len(ldf),
                         activities=int(ldf["concept:name"].nunique()),
                         mean_len=round(float(tl.mean()), 1))
                try:
                    t0 = time.time(); net, im, fm = discover(log, miner)
                    r.update(disc_s=round(time.time() - t0, 1),
                             places=len(net.places), transitions=len(net.transitions),
                             arcs=len(net.arcs),
                             silent=sum(1 for x in net.transitions if x.label is None),
                             simplicity=round(sim_eval.apply(net), 4))
                    r["fitness"] = round(fit_eval.apply(
                        log, net, im, fm, variant=fit_eval.Variants.TOKEN_BASED,
                        parameters=P)["log_fitness"], 4)
                    r["generalization"] = round(gen_eval.apply(log, net, im, fm), 4)
                    try:
                        t0 = time.time()
                        r["precision"] = round(prec_eval.apply(
                            log, net, im, fm,
                            variant=prec_eval.Variants.ETCONFORMANCE_TOKEN,
                            parameters=P), 4)
                        r["prec_s"] = round(time.time() - t0, 1)
                    except Exception as e:
                        r["precision"] = None
                        r["prec_note"] = f"{type(e).__name__}"
                    print(f"   {miner:<10} fit={r['fitness']:.4f} "
                          f"prec={r.get('precision')} gen={r['generalization']:.4f} "
                          f"simp={r['simplicity']:.4f} silent={r['silent']}", flush=True)
                except Exception as e:
                    r["error"] = f"{type(e).__name__}: {str(e)[:120]}"
                    print(f"   {miner:<10} FAILED {r['error']}", flush=True)
                results.append(r); flush()
            del log, ldf; gc.collect()
        if df is not None:
            del df
        gc.collect()

quality = pd.read_csv(OUT)
print(f"\n{len(quality)} rows -> {OUT}")
quality

## 4. Does precision change the ranking?

The key question. If Inductive Miner tops fitness but bottoms precision, the
original recommendation does not survive.

In [ ]:
q = quality[quality.error.isna()] if "error" in quality else quality
for notion in q.notion.unique():
    sub = q[q.notion == notion]
    print(f"\n=== {notion}-level: mean across faculty-role partitions ===")
    agg = sub.groupby("miner")[["fitness", "precision", "generalization",
                                "simplicity"]].mean().round(4)
    print(agg.to_string())
    if agg["precision"].notna().all():
        print(f"  best by fitness  : {agg['fitness'].idxmax()}")
        print(f"  best by precision: {agg['precision'].idxmax()}")
        if agg["fitness"].idxmax() != agg["precision"].idxmax():
            print("  -> RANKING REVERSES when precision is included")

## 5. LaTeX output

In [ ]:
for role in ["Lecturer", "Student"]:
    for notion in q.notion.unique():
        sub = q[(q.role == role) & (q.notion == notion)]
        if not len(sub):
            continue
        print(f"\n% ---- {role}, {notion}-level case notion ----")
        for fac in FACULTIES:
            r = sub[sub.faculty == fac].set_index("miner")
            if not len(r):
                continue
            def g(m, c):
                try:
                    v = r.loc[m, c]
                    return f"{v:.3f}" if pd.notna(v) else "--"
                except Exception:
                    return "--"
            cells = []
            for c_ in ["fitness", "precision", "generalization", "simplicity"]:
                cells += [g(m, c_) for m in ["alpha", "heuristic", "inductive"]]
            print(f"{fac} & " + " & ".join(cells) + r" \\")